In [1]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [3]:
from src.data.load_data import load_dataset
from src.config import TARGET_COLUMN

df = load_dataset()


In [4]:
LARGE_AGE_GAP = 10

LARGE_EDU_GAP = 2

# Data content

In [5]:
a_cols = sorted([col for col in df.columns if col.startswith("a_")])

rows = []

for col in a_cols:
    unique_vals = df[col].unique()
    n_unique = len(unique_vals)
    
    clean_vals = [
        v.item() if isinstance(v, (np.integer, np.floating)) else v
        for v in unique_vals
    ]
    
    try:
        clean_vals = sorted(clean_vals)
    except:
        pass
    
    rows.append({
        "column": col,
        "dtype": df[col].dtype,
        "n_unique": n_unique,
        "sample_values": clean_vals[:10]
    })

summary_df = pd.DataFrame(rows)

summary_df

,column,dtype,n_unique,sample_values
0,a_age,int64,38,"[18, 19, 20, 21, 22, 23, 24, 25, 26, 27]"
1,a_agreeableness,float64,101,"[0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]"
2,a_career_ambition,float64,101,"[0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]"
3,a_career_field,str,10,"[Creative Arts, Education, Engineering, Entrepreneurship, Finance, Healthcare, Law, Marketing, Science, Tech]"
4,a_chronotype,float64,101,"[0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]"
5,a_conscientiousness,float64,101,"[0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]"
6,a_education,int64,5,"[1, 2, 3, 4, 5]"
7,a_emotional_expressiveness,float64,101,"[0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]"
8,a_extraversion,float64,101,"[0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09]"
9,a_location,str,3,"[Rural, Suburban, Urban]"


# Build features based on difference between A and B

As previously seen, the raw data does not give a strong signal to help us predict the longevity of a relationship as it only considers data at individual level.

In this section we will engineer features based on the difference between the values of person A and person B in a same couple and check if these new features bring us more insights.

In [6]:
a_cols = [col for col in df.columns if col.startswith("a_")]
b_cols = [col for col in df.columns if col.startswith("b_")]

len(a_cols), len(b_cols)


(13, 13)

## SCORES (personality) related features


In [7]:
personality_traits = [
    "agreeableness",
    "career_ambition",
    "chronotype",
    "conscientiousness",
    "emotional_expressiveness",
    "extraversion",
    "openness",
    "spontaneity"
]

for trait in personality_traits :
    df[f"{trait}_diff"] = abs(df[f"a_{trait}"] - df[f"b_{trait}"])

For scores data type, we will consider the absolute value of the difference between A and B values for each raw data.

## CATEGORIES related features

In [8]:
categorical_traits = [
    "career_field",
    "location",
    "love_language"
]

for trait in categorical_traits:
    df[f"same_{trait}"] = (
        df[f"a_{trait}"] == df[f"b_{trait}"]
    ).astype(int)

## INTEGER (Age and education) related features

In [9]:
# Age features
df["age_diff"] = abs(df["a_age"] - df["b_age"])
df["large_age_gap"] = (df["age_diff"] >= LARGE_AGE_GAP).astype(int)
df["age_mean"] = (df["a_age"] + df["b_age"]) / 2


In [10]:

# Education features
df["education_diff"] = abs(df["a_education"] - df["b_education"])
df["large_education_gap"] = (df["education_diff"] >= LARGE_EDU_GAP).astype(int)

For these two data, the large age and education gap will be a deliberate choice (cf. CONSTANT). We can test several numbers but for a start 10 years will be considered a large age gap and 2 years a large educational gap.

# Check correlations of engineered features with target

In [11]:
# Identifier toutes les features construites
built_features = [
    col for col in df.columns
    if (
        "_diff" in col
        or "_avg" in col
        or "_mean" in col
        or col.startswith("same_")
        or col.startswith("large_")
        or col in ["mean_personality_diff", "personality_distance"]
    )
]

# Ajouter la target
engineered_columns = built_features + [TARGET_COLUMN]
engineered_df = df[engineered_columns].copy()
engineered_df.head()

,agreeableness_diff,career_ambition_diff,chronotype_diff,conscientiousness_diff,emotional_expressiveness_diff,extraversion_diff,openness_diff,spontaneity_diff,same_career_field,same_location,same_love_language,age_diff,large_age_gap,age_mean,education_diff,large_education_gap,relationship_longevity_months
0,0.35,0.54,0.18,0.01,0.14,0.17,0.32,0.19,0,0,0,26,1,33.0,0,0,60
1,0.31,0.14,0.02,0.14,0.11,0.39,0.07,0.16,0,0,0,6,0,35.0,1,0,59
2,0.15,0.02,0.34,0.39,0.21,0.57,0.05,0.17,0,1,0,21,1,35.5,2,1,84
3,0.26,0.15,0.37,0.60,0.06,0.07,0.13,0.25,0,0,0,18,1,29.0,2,1,70
4,0.18,0.07,0.54,0.30,0.06,0.04,0.09,0.27,0,0,1,14,1,29.0,1,0,68


In [12]:
print("Number of engineered features:", len(built_features))
print("Total columns (with target):", len(engineered_df.columns))

Number of engineered features: 16
Total columns (with target): 17


In [13]:
corr_df = engineered_df.corr()

target_corr = (
    corr_df["relationship_longevity_months"]
    .drop("relationship_longevity_months")  # retire l'auto-corrélation
    .to_frame(name="correlation_with_target")
)


target_corr["abs_correlation"] = target_corr["correlation_with_target"].abs()

target_corr = target_corr.sort_values(by="abs_correlation", ascending=False)

target_corr

,correlation_with_target,abs_correlation
same_love_language,0.204678,0.204678
career_ambition_diff,-0.152472,0.152472
emotional_expressiveness_diff,-0.141328,0.141328
spontaneity_diff,-0.094323,0.094323
openness_diff,-0.076087,0.076087
extraversion_diff,-0.069286,0.069286
chronotype_diff,-0.066272,0.066272
education_diff,-0.049569,0.049569
same_career_field,0.049495,0.049495
conscientiousness_diff,-0.046708,0.046708


# Save the FEATURES TABLE in a CSV

In [14]:
engineered_df.to_csv(
    "../src/data/processed/engineered_dataset.csv",
    index=False
)